# Lab — Retrieving Data via APIs and Web Scraping (Starter)

**Module 8 — APIs and Web Scraping**  
**Estimated time:** ~60 minutes  
**Tools:** Python 3, Jupyter Notebook, Requests, BeautifulSoup, Pandas

In this lab you will:
- Retrieve structured data from a REST API using `requests`
- (Optionally) demonstrate how authentication headers work
- Parse JSON into a Pandas DataFrame
- Scrape structured data from HTML with BeautifulSoup
- Store raw outputs in `data/raw/` following ethical and reproducible practices


## 1) Setup & Environment Check

In [ ]:
import os
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Optional: pretty print JSON
import json

print('pandas version:', pd.__version__)
print('requests version:', requests.__version__)

# Create expected folders (safe if they already exist)
Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('notebooks').mkdir(parents=True, exist_ok=True)

print('Folders ready: data/raw, data/processed, notebooks')


## 2) Retrieve Data from an API (Requests)

We'll use a public demo API that doesn't require a key:

- Endpoint: `https://jsonplaceholder.typicode.com/posts`

> Note: In real projects, you must follow the API's documentation and usage limits.


In [ ]:
api_url = 'https://jsonplaceholder.typicode.com/posts'

response = requests.get(api_url, timeout=30)
print('Status code:', response.status_code)

# TODO: Inspect headers if you want
# response.headers


In [ ]:
# TODO: Parse JSON safely
# If the status code isn't 200, handle it gracefully.

if response.ok:
    data = response.json()
    print('Records received:', len(data))
else:
    raise RuntimeError(f'Request failed with status {response.status_code}: {response.text[:200]}')


In [ ]:
# TODO: Save RAW API response to disk (reproducibility)
raw_api_path = Path('data/raw/api_posts.json')
raw_api_path.write_text(json.dumps(data, indent=2), encoding='utf-8')

print('Saved raw API data to:', raw_api_path)


### Optional: What does API authentication look like?

Many APIs require authentication (API keys, bearer tokens, etc.).  
Below is an example pattern using an environment variable. **Do not hardcode secrets in notebooks.**

You can skip this if your chosen API doesn't require auth.


In [ ]:
# OPTIONAL (example only): auth header pattern
# export API_TOKEN='your_token_here'  (macOS/Linux)
# setx API_TOKEN 'your_token_here'    (Windows PowerShell)

api_token = os.getenv('API_TOKEN')

headers = {}
if api_token:
    headers = {'Authorization': f'Bearer {api_token}'}
    print('Auth header prepared (token not printed).')
else:
    print('No API_TOKEN found. Skipping authenticated request example.')

# Example:
# secure_response = requests.get('https://api.example.com/secure-endpoint', headers=headers)


## 3) Extract and Structure API Data (Pandas)

In [ ]:
# TODO: Convert API JSON into a DataFrame
df_posts = pd.DataFrame(data)
df_posts.head()


In [ ]:
# TODO: Select relevant fields (keep it small and analysis-friendly)
# Example: userId, id, title
df_posts_subset = df_posts[['userId', 'id', 'title']].copy()
df_posts_subset.head()


In [ ]:
# TODO: Save a structured snapshot (still considered 'raw' if untransformed)
raw_csv_path = Path('data/raw/api_posts_subset.csv')
df_posts_subset.to_csv(raw_csv_path, index=False)
print('Saved structured API snapshot to:', raw_csv_path)


## 4) Web Scraping with BeautifulSoup

For a stable lab experience, we'll scrape a **local HTML file** instead of relying on a live website.

- File: `data/raw/sample_page.html`

If the file doesn't exist yet, the next cell will generate a sample HTML page for you.


In [ ]:
html_path = Path('data/raw/sample_page.html')

if not html_path.exists():
    sample_html = '''
    <!doctype html>
    <html>
      <head><title>Sample Products Page</title></head>
      <body>
        <h1>Products</h1>
        <ul class="products">
          <li class="product">
            <span class="name">Notebook Stand</span>
            <span class="price" data-currency="EUR">19.99</span>
          </li>
          <li class="product">
            <span class="name">Wireless Mouse</span>
            <span class="price" data-currency="EUR">24.50</span>
          </li>
          <li class="product">
            <span class="name">USB-C Hub</span>
            <span class="price" data-currency="EUR">34.00</span>
          </li>
        </ul>
      </body>
    </html>
    '''
    html_path.write_text(sample_html.strip(), encoding='utf-8')
    print('Created:', html_path)
else:
    print('Found existing:', html_path)


In [ ]:
# TODO: Load + parse HTML
html_text = html_path.read_text(encoding='utf-8')
soup = BeautifulSoup(html_text, 'html.parser')

# Inspect quickly
soup.title.text


In [ ]:
# TODO: Extract structured data (names + prices)
products = []

for item in soup.select('li.product'):
    name = item.select_one('.name').get_text(strip=True)
    price_text = item.select_one('.price').get_text(strip=True)
    currency = item.select_one('.price').get('data-currency', 'EUR')
    products.append({'name': name, 'price': float(price_text), 'currency': currency})

products[:3], len(products)


In [ ]:
# TODO: Convert to DataFrame and save raw scrape results
df_products = pd.DataFrame(products)
df_products


In [ ]:
scrape_path = Path('data/raw/scraped_products.csv')
df_products.to_csv(scrape_path, index=False)
print('Saved scraped results to:', scrape_path)


## 5) Ethical and Legal Considerations (Reflection)

In a markdown cell below, answer:

- When should an API be preferred over web scraping?
- What should you check before scraping a real website?
- How can you reduce impact on a website when scraping?


## 6) Deliverable Check

You should now have created:

- `data/raw/api_posts.json` (raw API response)
- `data/raw/api_posts_subset.csv` (structured snapshot)
- `data/raw/sample_page.html` (local HTML used for scraping)
- `data/raw/scraped_products.csv` (scraped structured data)

Next: commit and push your work to GitHub.


## Version Control

In [ ]:
# Run these commands in your terminal (not inside Python):

# git init
# git add .
# git commit -m "Retrieve data via APIs and web scraping"
# git branch -M main
# git push -u origin main
